# The Cajeta notebook kernel — a runnable tour

Every cell below runs offline against a local repository that
`./setup.sh` stages. Nothing here touches the network.

**Run `./setup.sh` first**, then start Jupyter from this project so the
kernel adopts it, with the tour's trust store on the environment:

```
./setup.sh
CAJETA_TRUST_KEYS_DIR="$PWD/trust" jupyter lab notebooks/tour.ipynb
```

Run the cells in order. **Six of them are meant to fail** — they are the
point, not an accident, and each is labelled. After every one of them the
kernel keeps serving and the session keeps its bindings; that is the
property being demonstrated.


## 1 — The session accumulates

A cell is a script unit. Bindings and classes made in one cell are still
there in the next; only the code you run is new.


In [ ]:
int32 a = 20;
a + 22;


In [ ]:
public class Counter {
    public int32 n;
    public Counter(int32 start) { this.n = start; }
    public int32 next() { this.n = this.n + 1; return this.n; }
}

Counter c = heap Counter(a);
c.next();


In [ ]:
// `a` and `c` are both still here, and `c` kept its state.
c.next() + a;


## 2 — Installing a library mid-session

`Packages.install` acquires a library into the *running* kernel. Watch
the phases stream below the cell: resolving, verifying, splicing. A
fetch is never a silent stall.

It returns the resolved version.


In [ ]:
import cajeta.session.Packages;

Packages.install("demo", "1.*");


Acquisition and binding are separate. `install` put the archive on the
classpath; `import` is what binds a name — **in a later cell**, which is
the next section's subject.


In [ ]:
import demo.Stats;

Stats.sum(20, 22);


### Re-running is safe

Installing something already loaded at a satisfying version is a no-op
that returns the loaded version, so running a notebook top to bottom a
second time re-fetches nothing.


In [ ]:
Packages.install("demo", "1.*");


## 3 — What a failure costs you

**The next two cells are expected to fail.** A notebook session can
represent hours of work, so a bad install costs you the install and
nothing else.


**Expected to fail** — the loaded version is 1.0.0, and `2.*` excludes
it. Installs are additive: JIT'd code from the loaded copy may be live,
so a session cannot swap it out. The error names both versions and says
a restart is what changing versions takes.


In [ ]:
Packages.install("demo", "2.*");


**Expected to fail** — no library called `ghost` exists. The error
names the constraint *and* every repository consulted, so you know
whether to fix the name, fix the constraint, or add a repository.


In [ ]:
Packages.install("ghost", "1.*");


Both failed, and the session is untouched — same `a`, same `c`, same
`Stats`:


In [ ]:
c.next() + a + Stats.answer();


### The one rule to remember

**A cell cannot import what it installs.** The cell is compiled before
its code runs, so the import is resolved before the install has
happened. This is not a limitation to work around; it falls out of what
a cell is.

**Expected to fail** — and the diagnostic says exactly this.


In [ ]:
import plain.Plain;

Packages.install("plain", "1.*");
Plain.value();


Following the hint — install in one cell:


In [ ]:
Packages.install("plain", "1.*");


...and import in the next:


In [ ]:
import plain.Plain;

Plain.value();


## 4 — Verification

Installing code into a live session is a supply-chain surface, so an
archive is checked before it is spliced.

Its sha256 is verified against the checksum the repository **publishes**
— not against a hash of the bytes just downloaded, which would only
prove the download was self-consistent.

`plain` above carries no signature and installed anyway: signatures are
opportunistic by default. A checksum authenticates the *mirror*; only a
signature says a publisher you trust stands behind the bytes. To make
them mandatory, add `"require-signatures": true` to `cajeta.json` and
re-run — that install then fails.

`demo` in Part 2 *was* signed, by a key `setup.sh` put in the trust
store this kernel was pointed at.

**Expected to fail** — `rogue` carries a perfectly valid ed25519
signature, made by a key this machine does not trust. That is the case
that matters: the bytes verify against *a* key, just not one you chose.


In [ ]:
Packages.install("rogue", "1.*");


Trust is decided by the machine, never by the repository — an archive
cannot vouch for itself by shipping a key alongside. Manage the keys
with `cajeta trust add` / `cajeta trust list`.

A signed archive on a machine with **no** trusted keys is also refused:
"signed but uncheckable" is not the same as fine. (Restart this kernel
without `CAJETA_TRUST_KEYS_DIR` and Part 2 fails that way.)


## 5 — Keeping the dependency

An install lasts as long as the session. The manifest is the
reproducible record, so a notebook you intend to share should record
what it needs.

`installAndSave` installs exactly as `install` does *and* writes the
dependency into `cajeta.json`, through the same format-preserving editor
`cajeta add` uses — your comments and layout survive. It is a separate
method rather than a flag so that reading the call tells you a file was
written.

`demo` is already loaded, so the install half is a no-op — but the save
still happens, which is the point:


In [ ]:
Packages.installAndSave("demo", "1.*");


Look at `cajeta.json` now: `settings.dependencies` has a `demo` entry,
and every comment in the file is still there. Restart the kernel and
`import demo.Stats` works in the *first* cell, with no install call.

`git checkout cajeta.json` puts it back.


## 6 — An install never shadows your session

If an archive declares a class the session already holds, the install is
refused. Your cell's definition wins — installed code cannot quietly
replace something you defined.

First, define a class named `Marker`:


In [ ]:
public class Marker {
    public static int32 value() { return 1; }
}

Marker.value();


**Expected to fail** — the `coll` library declares `coll.Marker`, and
the error says *which* kind of collision this is: declared by an earlier
cell, not by another archive.


In [ ]:
Packages.install("coll", "1.*");


Your `Marker` is untouched:


In [ ]:
Marker.value();


## Where to go next

- Add `"require-signatures": true` to `cajeta.json` and re-run Part 3's
  `plain` install — the unsigned archive is now refused.
- Restart without `CAJETA_TRUST_KEYS_DIR` and re-run Part 2 — a signed
  archive with no trusted keys is refused rather than waved through.
- Run `./setup.sh --clean` to remove the staged repository and keys.

Reference: [the notebooks guide](../../../docs/guide/24-notebooks.md) and
[`cajeta.session.Packages`](../../../docs/stdlib/session/Packages.md).
